# ParkCast Vision — Week 2: Cross-lot 도메인 갭 평가

`parkcast/domain.py` 모듈을 불러와 실행한 결과임.

**핵심 질문**: Week 1의 mAP50 0.9944는 진짜 일반화 성능인가, 아니면 random split의
data leakage(같은 시점 사진이 train/test에 섞여 들어감) 때문인가?

**방법**: 라벨 없이 ResNet50 임베딩 + K-Means로 주차장을 자동 발견하고, 같은 데이터를
Random(Week1 결과 재사용) / Date / Lot 세 가지 split으로 나눠 mAP를 비교함.

> ⚠️ **결과 요약**: 이 실험은 네거티브 결과였음. 세 split의 mAP50이 거의 동일하게 나온
> 것은 도메인 갭이 없어서가 아니라, K-Means 클러스터링이 PKLot의 실제 주차장(3개)을
> 제대로 분리하지 못했기 때문임(best_k=6로 실제보다 많이 쪼개짐). 다만 Date split의
> mAP50-95 하락은 유의미한 신호로 확인됨. 자세한 해석은 이 노트북 맨 아래 및
> [README.md](../README.md)의 "Cross-lot 도메인 갭 평가 결과 (Week 2)" 섹션 참조.

이 노트북을 다시 실행하면 K-Means/학습의 미세한 무작위성으로 정확히 같은 소수점
숫자가 안 나올 수 있음(구조적 결론은 동일할 것으로 예상). 아래 마크다운에 적힌
수치는 최초 실행 시점에 기록된 값임.


## 0. 환경 확인

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-"}')


In [ ]:
!pip install -q ultralytics scikit-learn umap-learn


## 1. 저장소 준비 (parkcast 패키지 import)

`parkcast/domain.py`, `parkcast/train.py`, `parkcast/evaluate.py`를 그대로 불러와 씀 —
이 셀에서 리포지토리가 이미 있으면(같은 세션에서 Week1/Week3 노트북을 먼저 돌렸다면)
다시 클론하지 않음.


In [ ]:
import os, sys

REPO_URL = 'https://github.com/kth020829-cell/Competition.git'
REPO_DIR = '/content/Competition'
PARKCAST_DIR = f'{REPO_DIR}/2025 국토교통 데이터활용 경진대회/parkcast'

if not os.path.exists(PARKCAST_DIR):
    !git clone --depth 1 "{REPO_URL}" "{REPO_DIR}"
else:
    print('리포지토리 이미 존재:', REPO_DIR)

sys.path.insert(0, PARKCAST_DIR)
import parkcast
print('parkcast 로드 위치:', parkcast.__file__)


## 2. Drive 마운트 + 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/ParkCast'
MODEL_DIR = f'{PROJECT_ROOT}/models'
RESULTS_DIR = f'{PROJECT_ROOT}/results'
WEEK2_DIR = f'{RESULTS_DIR}/cross_lot'
os.makedirs(WEEK2_DIR, exist_ok=True)

# 데이터는 로컬(/content)에 두고 "결과만" Drive에 저장하는 방식이 세션 중 가장 안정적이었음
# (Drive 직접 접근은 느리고, Drive 경로에 symlink를 걸면 종종 실패함)
DATA_ROOT = '/content'
YOLO_ROOT = '/content/pklot_yolo'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('WEEK2_DIR:   ', WEEK2_DIR)


## 3. 데이터 준비 (PKLot raw + YOLO 변환본)

Week 1 노트북에서 이미 받아 Drive에 백업해둔 `pklot-dataset.zip`을 재사용함. YOLO
변환본(`pklot_yolo/`)이 로컬에 없으면 `parkcast.data.coco_to_yolo`로 다시 만듦
(라벨 파일이라 몇 초면 끝남 — 무거운 건 이미지 심링크/복사뿐).


In [ ]:
if not os.path.exists(f'{DATA_ROOT}/train/_annotations.coco.json'):
    backup = f'{PROJECT_ROOT}/pklot-dataset.zip'
    assert os.path.exists(backup), (
        f'백업 zip이 {backup}에 없음 — ParkCast_Week1_YOLOv8.ipynb를 먼저 실행해 '
        f'데이터를 받고 Drive에 백업해둬야 함.'
    )
    print('PKLot raw 데이터 복구 중...')
    !cp "{backup}" /content/
    !cd /content && unzip -q pklot-dataset.zip
else:
    print('PKLot raw 데이터 이미 존재:', DATA_ROOT)

if not os.path.exists(f'{YOLO_ROOT}/data.yaml'):
    print('YOLO 포맷(detect) 변환 중...')
    from parkcast.data import coco_to_yolo
    coco_to_yolo(DATA_ROOT, YOLO_ROOT)
else:
    print('YOLO 변환본 이미 존재:', YOLO_ROOT)


## 4. 이미지 메타데이터 수집 (파일명에서 날짜 파싱)

Roboflow 파일명 패턴(`2013-03-22_12_55_08_jpg.rf.해시.jpg`)에서 날짜/시간을 추출함
(`parkcast.domain.build_image_metadata`). 날짜가 있어야 Date split을 만들 수 있음.


In [ ]:
from parkcast.domain import build_image_metadata

meta = build_image_metadata(YOLO_ROOT)
print(f'총 {len(meta):,}장, 날짜 파싱 성공 {meta["date"].notna().sum():,}장')
meta.head()


## 5. ResNet50 임베딩 추출 + K-Means로 주차장 자동 발견

파일명에 주차장 정보가 없으므로, ImageNet pretrained ResNet50 임베딩(2048-dim, avgpool
직전)을 PCA(50) → K-Means(k=2~6, silhouette 최댓값 선택)로 클러스터링해 "주차장"을
라벨 없이 간접 추정함. PKLot의 실제 주차장은 3개(PUCPR/UFPR04/UFPR05)임.


In [ ]:
from parkcast.domain import extract_resnet50_embeddings, discover_lots

embeddings = extract_resnet50_embeddings(meta['path'].tolist())
cluster = discover_lots(embeddings, with_umap=False)
meta['lot_cluster'] = cluster.labels

print(f'best_k = {cluster.best_k}  (참고: 실제 물리적 주차장 수는 3개)')
print(f'silhouette(best_k) = {cluster.silhouette_scores[cluster.best_k]:.4f}')
print()
for k, s in sorted(cluster.silhouette_scores.items()):
    print(f'  k={k}: silhouette={s:.4f}')
print()
print('클러스터별 이미지 수:')
print(meta['lot_cluster'].value_counts().sort_index())


> **실행 결과**: `best_k=6`, `silhouette=0.2347`. 실제 주차장(3개)보다 클러스터가 많이
> 나왔고 silhouette도 낮음(1에 가까울수록 경계가 뚜렷한데 0.23은 약함) — 임베딩이 주차장
> 정체성보다 조명·구도 같은 저수준 특징에 더 지배됐을 가능성을 시사함. 이게 이후 결과
> 해석의 핵심 단서가 됨(맨 아래 참조).

## 6. Date split / Lot split 구성

- **Date split**: 날짜 기준 시간순 80% train / 10% val / 10% test
- **Lot split**: 표본이 가장 적은 클러스터를 통째로 test로 떼어내 "본 적 없는 주차장"으로
  평가(의도상). 5번 셀의 클러스터링 품질에 따라 실제로 도메인이 분리됐는지가 갈림.

기존 `pklot_yolo/`의 라벨 파일을 재사용해서(다시 만들 필요 없이) 이미지/라벨만 새
split 폴더로 재배치함(`parkcast.domain.build_yolo_split`).


In [ ]:
from parkcast.domain import assign_date_split, assign_lot_split, build_yolo_split
from parkcast.utils import YOLO_CLASS_NAMES

meta['split_date'] = assign_date_split(meta)
meta['split_lot'] = assign_lot_split(meta, cluster_col='lot_cluster')
meta.to_csv(f'{WEEK2_DIR}/meta_with_splits.csv', index=False)

print('Date split:')
print(meta['split_date'].value_counts())
print()
print('Lot split:')
print(meta['split_lot'].value_counts())


In [ ]:
date_yaml = build_yolo_split(meta, 'split_date', YOLO_ROOT, f'{DATA_ROOT}/yolo_date', YOLO_CLASS_NAMES)
lot_yaml = build_yolo_split(meta, 'split_lot', YOLO_ROOT, f'{DATA_ROOT}/yolo_lot', YOLO_CLASS_NAMES)

print('date_yaml:', date_yaml)
print('lot_yaml: ', lot_yaml)


> **실행 결과**: Lot split 크기는 train 9,842 / val 1,093 / test 1,481.

## 7. Date split 학습 + 평가

`configs/default.yaml`의 학습 설정을 그대로 재사용(모델만 바뀐 데이터로 처음부터 학습).


In [ ]:
from parkcast.train import train_yolo
from parkcast.evaluate import evaluate_on_test
from parkcast.utils import load_config, set_seed

set_seed(42)
cfg = load_config(f'{PARKCAST_DIR}/configs/default.yaml')
train_cfg = cfg.train.raw

date_best = train_yolo(date_yaml, WEEK2_DIR, {**train_cfg, 'run_name': 'split_date'})
date_metrics = evaluate_on_test(
    date_best, date_yaml, WEEK2_DIR, run_name='split_date_test',
    imgsz=train_cfg['imgsz'], batch=train_cfg['batch'], class_names=YOLO_CLASS_NAMES,
)
print('Date split test 결과:', date_metrics)


## 8. Lot split 학습 + 평가

"본 적 없는 주차장"으로 일반화되길 기대했던 split. 5번 셀에서 본 클러스터링 품질이
안 좋았기 때문에 결과 해석에 주의가 필요함(9번 셀 참조).


In [ ]:
lot_best = train_yolo(lot_yaml, WEEK2_DIR, {**train_cfg, 'run_name': 'split_lot'})
lot_metrics = evaluate_on_test(
    lot_best, lot_yaml, WEEK2_DIR, run_name='split_lot_test',
    imgsz=train_cfg['imgsz'], batch=train_cfg['batch'], class_names=YOLO_CLASS_NAMES,
)
print('Lot split test 결과:', lot_metrics)


## 9. Random vs Date vs Lot 비교 — 결과와 해석

Random split 결과는 Week 1에서 이미 구한 값을 그대로 가져다 씀(다시 학습할 필요 없음).


In [ ]:
from parkcast.evaluate import EvalMetrics
from parkcast.domain import compare_splits, plot_split_comparison

random_metrics = EvalMetrics(
    mAP50=0.9944, mAP50_95=0.9886, precision=0.9977, recall=0.9975, per_class_mAP50={},
)

results = {'random': random_metrics, 'date': date_metrics, 'lot': lot_metrics}
df = compare_splits(results)
df.to_csv(f'{WEEK2_DIR}/split_comparison.csv', index=False)
plot_split_comparison(df, save_path=f'{WEEK2_DIR}/split_comparison.png')
df


### 실행 결과 (test set)

| Split | mAP50 | mAP50-95 | Precision | Recall |
|-------|-------|----------|-----------|--------|
| Random | 0.9944 | 0.9886 | 0.9977 | 0.9975 |
| Date   | 0.995  | 0.805  | ~0.995 | ~0.995 |
| Lot    | 0.995  | 0.989  | 0.999  | 0.998  |

(Date/Lot의 정확한 소수점은 위 9번 셀이 저장하는 `{WEEK2_DIR}/split_comparison.csv` 참조 —
여기 표는 실행 로그에서 옮긴 값)

### 이 결과의 의미 — 네거티브 결과임, "도메인 갭 없음"이 아니라 "분리가 안 됨"

세 split의 mAP50이 전부 0.99대로 거의 동일한 것은 **도메인 갭이 없어서가 아니라, K-Means
클러스터링이 물리적 주차장을 제대로 분리하지 못했기 때문**임. 5번 셀에서 본 `best_k=6`이
실제 주차장 수(3)보다 많이 나온 것 자체가 그 증거 — 같은 주차장이 날씨·조명별로 서로 다른
클러스터로 쪼개진 것으로 보임. 그 결과 "Lot split"의 train/val/test에 사실상 같은 주차장이
섞여 들어가, 의도와 달리 또 하나의 random split이 되어버렸고, 이게 mAP50이 떨어지지 않은
진짜 이유임.

다만 **Date split의 mAP50-95가 0.9886 → 0.805로 떨어진 것은 유의미한 신호**임. mAP50-95는
IoU 기준이 엄격한 구간까지 평균낸 지표라, 시간적으로 학습/테스트를 분리하면(다른 날짜) 박스
위치를 정밀하게 맞추는 능력이 나빠진다는 걸 보여줌 — random split이 갖고 있던 leakage(같은
시점 사진이 train/test 양쪽에 들어가는 것)를 부분적으로 제거한 효과로 해석됨.

**결론**: 이 실험은 "cross-domain 검증에 성공했다"는 게 아니라, **임베딩 기반 비지도 도메인
분리 방법의 한계**를 보여준 네거티브 결과임. mAP 0.99가 leakage 때문일 수 있다는 의심을
실제로 검증하려 했고, 그 과정에서 클러스터링이 실패했다는 것 자체를 진단해냈고, Date
split을 통해 최소한 시간적 leakage의 영향은 정량적으로 확인했음. 더 lot-특이적인 피처
(예: 고정 배경 영역만 crop한 임베딩)로 클러스터링을 다시 시도하면 실제 주차장 3개로 더
깨끗하게 분리될 가능성이 있음.
